# Module 3.2 — Physics-Informed Neural Networks (Full Implementation)

Complete PyTorch PINN implementation for solving PDEs.

## Requirements
```
pip install torch matplotlib numpy
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Problem Setup: 1D Burgers Equation

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

Domain: $x \in [-1, 1]$, $t \in [0, 1]$

Initial condition: $u(x, 0) = -\sin(\pi x)$

Boundary conditions: $u(-1, t) = u(1, t) = 0$

Viscosity: $\nu = 0.01/\pi$

In [ ]:
nu = 0.01 / np.pi

class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.activation = nn.Tanh()
        self.linears = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.linears.append(nn.Linear(layers[i], layers[i+1]))
            # Xavier initialization
            nn.init.xavier_normal_(self.linears[-1].weight)
            nn.init.zeros_(self.linears[-1].bias)

    def forward(self, x, t):
        X = torch.cat([x, t], dim=1)
        for i, linear in enumerate(self.linears[:-1]):
            X = self.activation(linear(X))
        return self.linears[-1](X)

# Network: 2 inputs (x,t) -> 4 hidden layers of 50 -> 1 output (u)
layers = [2, 50, 50, 50, 50, 1]
model = PINN(layers).to(device)

## 2. Generate Collocation Points and Boundary Data

In [ ]:
# Collocation points inside domain
N_f = 10000
x_f = torch.rand(N_f, 1) * 2 - 1  # x in [-1, 1]
t_f = torch.rand(N_f, 1)           # t in [0, 1]
x_f = x_f.to(device).requires_grad_(True)
t_f = t_f.to(device).requires_grad_(True)

# Initial condition points
N_ic = 200
x_ic = torch.linspace(-1, 1, N_ic).reshape(-1, 1).to(device)
t_ic = torch.zeros(N_ic, 1).to(device)
u_ic = -torch.sin(np.pi * x_ic)  # u(x,0) = -sin(pi*x)

# Boundary condition points
N_bc = 100
t_bc = torch.linspace(0, 1, N_bc).reshape(-1, 1).to(device)
x_bc_left = -torch.ones(N_bc, 1).to(device)
x_bc_right = torch.ones(N_bc, 1).to(device)
u_bc = torch.zeros(N_bc, 1).to(device)

## 3. Define the Physics Loss

The PDE residual is computed using automatic differentiation:

$$\mathcal{L}_{\text{PDE}} = \frac{1}{N_f}\sum_{i=1}^{N_f}\left|u_t + u \cdot u_x - \nu u_{xx}\right|^2$$

In [ ]:
def pde_residual(model, x, t):
    u = model(x, t)
    
    # First derivatives
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u),
                              create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u),
                              create_graph=True)[0]
    
    # Second derivative
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x),
                               create_graph=True)[0]
    
    # Burgers equation residual
    residual = u_t + u * u_x - nu * u_xx
    return residual

def total_loss(model):
    # PDE residual loss
    res = pde_residual(model, x_f, t_f)
    loss_pde = (res ** 2).mean()
    
    # Initial condition loss
    u_pred_ic = model(x_ic, t_ic)
    loss_ic = ((u_pred_ic - u_ic) ** 2).mean()
    
    # Boundary condition loss
    u_pred_left = model(x_bc_left, t_bc)
    u_pred_right = model(x_bc_right, t_bc)
    loss_bc = ((u_pred_left - u_bc) ** 2).mean() + ((u_pred_right - u_bc) ** 2).mean()
    
    return loss_pde + 10 * loss_ic + 10 * loss_bc, loss_pde, loss_ic, loss_bc

## 4. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

n_epochs = 10000
history = {'total': [], 'pde': [], 'ic': [], 'bc': []}

for epoch in range(n_epochs):
    optimizer.zero_grad()
    loss, loss_pde, loss_ic, loss_bc = total_loss(model)
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    history['total'].append(loss.item())
    history['pde'].append(loss_pde.item())
    history['ic'].append(loss_ic.item())
    history['bc'].append(loss_bc.item())
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}: Total={loss.item():.6f}, '
              f'PDE={loss_pde.item():.6f}, IC={loss_ic.item():.6f}, BC={loss_bc.item():.6f}')

# Plot training history
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for key in history:
    ax.semilogy(history[key], label=key)
ax.legend(); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('PINN Training History'); ax.grid(True, alpha=0.3)
plt.show()

## 5. Visualization of Solution

In [ ]:
# Create evaluation grid
x_test = torch.linspace(-1, 1, 200).reshape(-1, 1)
t_test = torch.linspace(0, 1, 100).reshape(-1, 1)
X, T = torch.meshgrid(x_test.squeeze(), t_test.squeeze(), indexing='ij')

x_flat = X.reshape(-1, 1).to(device)
t_flat = T.reshape(-1, 1).to(device)

with torch.no_grad():
    u_pred = model(x_flat, t_flat).cpu().reshape(200, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Solution surface
c = axes[0].pcolormesh(T.numpy(), X.numpy(), u_pred.numpy(), cmap='RdBu_r', shading='auto')
plt.colorbar(c, ax=axes[0])
axes[0].set_xlabel('t'); axes[0].set_ylabel('x')
axes[0].set_title('PINN Solution u(x,t)')

# Snapshots at different times
for i, t_val in enumerate([0.0, 0.25, 0.5, 0.75, 1.0]):
    t_idx = int(t_val * 99)
    axes[1].plot(x_test.numpy(), u_pred[:, t_idx].numpy(), label=f't={t_val}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('u')
axes[1].legend(); axes[1].set_title('Solution Snapshots')

# PDE residual magnitude
x_res = torch.linspace(-1, 1, 100).reshape(-1, 1).to(device).requires_grad_(True)
t_res = torch.linspace(0, 1, 50).reshape(-1, 1).to(device).requires_grad_(True)
Xr, Tr = torch.meshgrid(x_res.squeeze(), t_res.squeeze(), indexing='ij')
xr_flat = Xr.reshape(-1, 1).requires_grad_(True)
tr_flat = Tr.reshape(-1, 1).requires_grad_(True)
res = pde_residual(model, xr_flat, tr_flat).detach().cpu().reshape(100, 50)
c2 = axes[2].pcolormesh(Tr.detach().numpy(), Xr.detach().numpy(), 
                         np.abs(res.numpy()), cmap='hot', shading='auto')
plt.colorbar(c2, ax=axes[2])
axes[2].set_xlabel('t'); axes[2].set_ylabel('x')
axes[2].set_title('|PDE Residual|')

plt.tight_layout()
plt.show()

## 6. Inverse Problem: Recovering Viscosity

Given solution data, we can also learn the viscosity parameter $\nu$.

In [ ]:
class PINN_Inverse(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.activation = nn.Tanh()
        self.linears = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.linears.append(nn.Linear(layers[i], layers[i+1]))
            nn.init.xavier_normal_(self.linears[-1].weight)
            nn.init.zeros_(self.linears[-1].bias)
        # Learnable viscosity parameter
        self.nu = nn.Parameter(torch.tensor(0.1))  # Initial guess

    def forward(self, x, t):
        X = torch.cat([x, t], dim=1)
        for linear in self.linears[:-1]:
            X = self.activation(linear(X))
        return self.linears[-1](X)

model_inv = PINN_Inverse(layers).to(device)
print(f'True nu = {nu:.6f}')
print(f'Initial guess = {model_inv.nu.item():.6f}')
print('Training would recover nu from solution data...')